# Terminator Paradox — Computational Demonstration Notebook
### FAIR Backward Induction Engine for Extensive-Form Games

This notebook accompanies the paper  
**“The Terminator Paradox: When Optimal Strategies Destroy Their Own Foundations.”**

It provides a fully generic, FAIR implementation of **backward induction** for finite extensive-form games with perfect information.  
The goal is to make the theoretical results of the paper **transparent, reproducible, and easy to modify**.

---

## Contents of this notebook

- **A generic backward-induction solver**  
  Works for any extensive-form game you define.

- **Three reproducible experiments**  
  1. Simplified Terminator game  
  2. Four-node centipede game  
  3. Endogenous game tree showing a *self-invalidating equilibrium*

- **ASCII visualization of game trees**  
  So you can inspect the structure of each game at a glance.

- **A template for defining your own game**  
  You can easily build new game trees by editing a few lines.

---

## How to use this notebook

### Running the examples
Execute the cells in order.  
Each experiment prints:
- the game tree  
- the backward-induction payoff  
- the equilibrium strategy  

### Defining your own game
Scroll to the section **“How to define your own game”**.  
There you will find a ready-to-edit function:

build_custom_game()


Modify:
- terminal nodes  
- internal nodes  
- actions  
- players  

Then run:

game = build_custom_game()
payoff, strategy = backward_induction(game.root)



---

## FAIR Principles

This implementation is:

- **Findable** — clear structure and documented functions  
- **Accessible** — pure Python, no dependencies  
- **Interoperable** — generic representation of game trees  
- **Reusable** — works for any extensive-form game  

---

Now, let’s move forward. Just remember: in this notebook, *the future is not set*.




In [1]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple


# ================================================================
# Basic Types
# ================================================================

Player = int
Action = str
Payoff = Tuple[float, ...]


# ================================================================
# Node and Game Representation
# ================================================================

@dataclass
class Node:
    """
    A node in an extensive-form game tree.

    Attributes:
        id: Unique identifier for the node.
        player: The player who moves at this node (None if terminal).
        children: Mapping from actions to successor nodes.
        payoff: Payoff vector if terminal, otherwise None.
    """
    id: int
    player: Optional[Player]
    children: Dict[Action, "Node"]
    payoff: Optional[Payoff] = None

    def is_terminal(self) -> bool:
        """Return True if the node is a terminal node."""
        return self.payoff is not None


@dataclass
class ExtensiveFormGame:
    """A complete extensive-form game with perfect information."""
    players: List[Player]
    root: Node


# ================================================================
# Backward Induction (Generic, FAIR)
# ================================================================

def backward_induction(node: Node) -> Tuple[Payoff, Dict[int, Action]]:
    """
    Compute the backward-induction solution from a given node.

    Returns:
        payoff: The equilibrium payoff vector from this node.
        strategy: A dictionary mapping node_id -> chosen action.

    This function is fully generic: it works for any game tree.
    """
    if node.is_terminal():
        return node.payoff, {}

    assert node.player is not None, "Non-terminal node must have a player."

    best_payoff = None
    best_action = None
    best_strategy: Dict[int, Action] = {}

    # Evaluate each action recursively
    for action, child in node.children.items():
        child_payoff, child_strategy = backward_induction(child)

        # Choose the action maximizing the current player's payoff
        if best_payoff is None or child_payoff[node.player] > best_payoff[node.player]:
            best_payoff = child_payoff
            best_action = action
            best_strategy = child_strategy

    # Record the chosen action at this node
    best_strategy = dict(best_strategy)
    best_strategy[node.id] = best_action

    return best_payoff, best_strategy


# ================================================================
# ASCII Tree Printer (for transparency and debugging)
# ================================================================

def print_tree(node: Node, indent: str = "", last: bool = True):
    """
    Pretty-print the game tree in ASCII form.

    Example output:
        Node 0 [Player 0]
        ├── Action 'Send'
        │   └── Node 1 [Terminal: payoff=(1,-1)]
        └── Action 'Wait'
            └── Node 2 [Terminal: payoff=(0,0)]
    """
    connector = "└── " if last else "├── "
    print(indent + connector + f"Node {node.id}", end="")

    if node.is_terminal():
        print(f"  [Terminal: payoff={node.payoff}]")
        return
    else:
        print(f"  [Player {node.player}]")

    indent += "    " if last else "│   "
    children = list(node.children.items())

    for i, (action, child) in enumerate(children):
        is_last = (i == len(children) - 1)
        print(indent + f"Action '{action}':")
        print_tree(child, indent, is_last)


# ================================================================
# 1. Simplified Terminator Game
# ================================================================

def build_terminator_game() -> ExtensiveFormGame:
    """Skynet chooses between Send and Wait."""
    send = Node(id=1, player=None, children={}, payoff=(+1.0, -1.0))
    wait = Node(id=2, player=None, children={}, payoff=(0.0, 0.0))

    root = Node(id=0, player=0, children={"Send": send, "Wait": wait})
    return ExtensiveFormGame(players=[0, 1], root=root)


# ================================================================
# 2. Four-Node Centipede Game (Backward Induction → TAKE)
# ================================================================

def build_centipede_game() -> ExtensiveFormGame:
    """A minimal centipede game where BI predicts TAKE at every node."""
    t0 = Node(id=10, player=None, children={}, payoff=(1.0, 0.0))
    t1 = Node(id=11, player=None, children={}, payoff=(0.0, 2.0))
    t2 = Node(id=12, player=None, children={}, payoff=(3.0, 1.0))
    t3 = Node(id=13, player=None, children={}, payoff=(2.0, 4.0))

    n3 = Node(id=3, player=1, children={"TAKE": t3})
    n2 = Node(id=2, player=0, children={"TAKE": t2, "PASS": n3})
    n1 = Node(id=1, player=1, children={"TAKE": t1, "PASS": n2})
    n0 = Node(id=0, player=0, children={"TAKE": t0, "PASS": n1})

    return ExtensiveFormGame(players=[0, 1], root=n0)


# ================================================================
# 3. Endogenous Game Tree (Self-Invalidating Equilibrium)
# ================================================================

def build_endogenous_example():
    """
    Build an endogenous game tree where executing the equilibrium
    destroys the subtree that made it optimal.
    """

    def tree_generator(strategy: Dict[int, Action]) -> ExtensiveFormGame:
        tA = Node(id=100, player=None, children={}, payoff=(2.0,))
        tC = Node(id=101, player=None, children={}, payoff=(3.0,))
        tBad = Node(id=102, player=None, children={}, payoff=(1.0,))
        tD = Node(id=103, player=None, children={}, payoff=(0.0,))

        # If strategy chooses B at root, the subtree disappears
        if strategy.get(0) == "B":
            root = Node(id=0, player=0, children={"A": tA, "B": tBad})
        else:
            n1 = Node(id=1, player=0, children={"C": tC, "D": tD})
            root = Node(id=0, player=0, children={"A": tA, "B": n1})

        return ExtensiveFormGame(players=[0], root=root)

    return tree_generator


def is_self_invalidating(tree_gen, strategy):
    """Return True if executing the strategy invalidates itself."""
    base_tree = tree_gen({})
    _, base_spe = backward_induction(base_tree.root)

    new_tree = tree_gen(strategy)
    _, new_spe = backward_induction(new_tree.root)

    return strategy != new_spe


# ================================================================
# HOW TO DEFINE YOUR OWN GAME
# ================================================================

def build_custom_game() -> ExtensiveFormGame:
    """
    Template for users who want to define their own game.

    Steps:
    1. Create terminal nodes (player=None, payoff=...).
    2. Create internal nodes with player indices and children.
    3. Return an ExtensiveFormGame with a root node.

    Modify freely for your experiments.
    """
    # Terminal nodes
    tL = Node(id=100, player=None, children={}, payoff=(0.0, 1.0))
    tX = Node(id=101, player=None, children={}, payoff=(2.0, 0.0))
    tY = Node(id=102, player=None, children={}, payoff=(1.0, 1.0))

    # Second-level node (Player 1)
    n1 = Node(id=1, player=1, children={"X": tX, "Y": tY})

    # Root node (Player 0)
    root = Node(id=0, player=0, children={"L": tL, "R": n1})

    return ExtensiveFormGame(players=[0, 1], root=root)


# ================================================================
# MAIN EXECUTION (Examples)
# ================================================================

def main():
    print("\n=== 1) Terminator Game ===")
    game = build_terminator_game()
    print_tree(game.root)
    payoff, strategy = backward_induction(game.root)
    print("BI payoff:", payoff)
    print("BI strategy:", strategy)

    print("\n=== 2) Centipede Game ===")
    game = build_centipede_game()
    print_tree(game.root)
    payoff, strategy = backward_induction(game.root)
    print("BI payoff:", payoff)
    print("BI strategy:", strategy)

    print("\n=== 3) Endogenous Game Tree ===")
    tree_gen = build_endogenous_example()
    base_tree = tree_gen({})
    print("Planning tree:")
    print_tree(base_tree.root)
    payoff, strategy = backward_induction(base_tree.root)
    print("Planning SPE:", strategy)

    print("\nChecking self-invalidating equilibrium...")
    print("Self-invalidating?", is_self_invalidating(tree_gen, strategy))

    # Uncomment to test your own game:
    # print("\n=== Custom Game ===")
    #game = build_custom_game()
    #print_tree(game.root)
    #payoff, strategy = backward_induction(game.root)
    #print("BI payoff:", payoff)
    #print("BI strategy:", strategy)
    
main()


=== 1) Terminator Game ===
└── Node 0  [Player 0]
    Action 'Send':
    ├── Node 1  [Terminal: payoff=(1.0, -1.0)]
    Action 'Wait':
    └── Node 2  [Terminal: payoff=(0.0, 0.0)]
BI payoff: (1.0, -1.0)
BI strategy: {0: 'Send'}

=== 2) Centipede Game ===
└── Node 0  [Player 0]
    Action 'TAKE':
    ├── Node 10  [Terminal: payoff=(1.0, 0.0)]
    Action 'PASS':
    └── Node 1  [Player 1]
        Action 'TAKE':
        ├── Node 11  [Terminal: payoff=(0.0, 2.0)]
        Action 'PASS':
        └── Node 2  [Player 0]
            Action 'TAKE':
            ├── Node 12  [Terminal: payoff=(3.0, 1.0)]
            Action 'PASS':
            └── Node 3  [Player 1]
                Action 'TAKE':
                └── Node 13  [Terminal: payoff=(2.0, 4.0)]
BI payoff: (1.0, 0.0)
BI strategy: {0: 'TAKE'}

=== 3) Endogenous Game Tree ===
Planning tree:
└── Node 0  [Player 0]
    Action 'A':
    ├── Node 100  [Terminal: payoff=(2.0,)]
    Action 'B':
    └── Node 1  [Player 0]
        Action 'C':
     